# 🚀 SARA & Standard RAG Baseline Checkpoint Audit & Establishment Notebook

This notebook provides a complete workflow to **audit, train, profile, and verify baseline model checkpoints** for the **QCCA (Query-Conditioned Evidence Allocation)** research project.

### Core Workflow Steps:
1. **System & GPU Metrics Audit**: Profile CPU RAM, GPU VRAM (NVIDIA TITAN RTX), PyTorch/CUDA environment.
2. **Training Configuration Audit**: Compare Standard RAG (`rag_qasper_lora_r16_seed42`) vs. SARA (`sara_qasper_proj_lr5e4_seed42`).
3. **Dataset & Checkpoint Verification**: Audit filesystem for dataset splits and adapter weights.
4. **Dataset Pipeline Setup**: Generate document-level disjoint QASPER splits if absent.
5. **Adapter Fine-Tuning Commands**: Launch training for Standard RAG and SARA adapters.
6. **Checkpoint Loader & Resource Profiler**: Measure peak VRAM/RAM, load time, and verify parameter norms.
7. **Inference & Generation Parity Sanity Test**: Test single-batch generation for loaded checkpoints.

In [1]:
import os
import sys
import time
import json
import psutil
import torch
import pandas as pd
from pathlib import Path

# Ensure repository root is in python path
REPO_ROOT = Path("..").resolve()
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

def print_system_metrics():
    ram = psutil.virtual_memory()
    print("=" * 60)
    print("SYSTEM RESOURCE & METRICS AUDIT")
    print("=" * 60)
    print(f"CPU RAM Total     : {ram.total / (1024**3):.2f} GB")
    print(f"CPU RAM Available : {ram.available / (1024**3):.2f} GB")
    print(f"CPU RAM Used      : {ram.used / (1024**3):.2f} GB ({ram.percent}%)")
    
    if torch.cuda.is_available():
        print(f"GPU Device        : {torch.cuda.get_device_name(0)}")
        total_vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        alloc_vram = torch.cuda.memory_allocated(0) / (1024**3)
        res_vram = torch.cuda.memory_reserved(0) / (1024**3)
        peak_vram = torch.cuda.max_memory_allocated(0) / (1024**3)
        print(f"GPU VRAM Total    : {total_vram:.2f} GB")
        print(f"GPU VRAM Allocated: {alloc_vram:.2f} GB")
        print(f"GPU VRAM Reserved : {res_vram:.2f} GB")
        print(f"GPU VRAM Peak     : {peak_vram:.2f} GB")
    else:
        print("GPU VRAM          : CUDA Not Available")
    print("=" * 60)

print_system_metrics()

SYSTEM RESOURCE & METRICS AUDIT
CPU RAM Total     : 62.47 GB
CPU RAM Available : 56.72 GB
CPU RAM Used      : 5.76 GB (9.2%)
GPU Device        : NVIDIA TITAN RTX
GPU VRAM Total    : 23.45 GB
GPU VRAM Allocated: 0.00 GB
GPU VRAM Reserved : 0.00 GB
GPU VRAM Peak     : 0.00 GB


## 1. Official Training Configurations Audit Matrix

In [2]:
import yaml

rag_config_path = REPO_ROOT / "config/language_modeling/Finetune_RAG_qasper_pubbase.yaml"
sara_config_path = REPO_ROOT / "config/language_modeling/Finetune_SARA_qasper_pubbase.yaml"

with open(rag_config_path) as f:
    rag_cfg = yaml.safe_load(f)
with open(sara_config_path) as f:
    sara_cfg = yaml.safe_load(f)

matrix_data = [
    ("Base Model", rag_cfg.get("model_name_or_path"), sara_cfg.get("model_name_or_path")),
    ("Output Checkpoint", rag_cfg.get("output_model_name"), sara_cfg.get("output_model_name")),
    ("Method Type", "Standard RAG (Projector-free)", "SARA (LoRA + Projector)"),
    ("Text Evidence (k)", rag_cfg.get("num_evidence"), sara_cfg.get("num_evidence")),
    ("Compressed Evidence (n-k)", rag_cfg.get("num_additional_evidence"), sara_cfg.get("num_additional_evidence")),
    ("LM LoRA LR", rag_cfg.get("model_learning_rate"), sara_cfg.get("model_learning_rate")),
    ("Projector LR", rag_cfg.get("projector_learning_rate"), sara_cfg.get("projector_learning_rate")),
    ("Freeze Projector", rag_cfg.get("freeze_projector"), sara_cfg.get("freeze_projector")),
    ("Per Device Batch Size", rag_cfg.get("per_device_train_batch_size"), sara_cfg.get("per_device_train_batch_size")),
    ("Gradient Accumulation Steps", rag_cfg.get("gradient_accumulation_steps"), sara_cfg.get("gradient_accumulation_steps")),
    ("Effective Global Batch", rag_cfg.get("per_device_train_batch_size") * rag_cfg.get("gradient_accumulation_steps"), sara_cfg.get("per_device_train_batch_size") * sara_cfg.get("gradient_accumulation_steps")),
    ("Max Train Steps", rag_cfg.get("max_train_steps"), sara_cfg.get("max_train_steps")),
    ("Seed", rag_cfg.get("seed"), sara_cfg.get("seed")),
]

df_matrix = pd.DataFrame(matrix_data, columns=["Hyperparameter", "Standard RAG Config", "SARA Config"])
display(df_matrix)

,Hyperparameter,Standard RAG Config,SARA Config
0,Base Model,mistralai/Mistral-7B-Instruct-v0.2,mistralai/Mistral-7B-Instruct-v0.2
1,Output Checkpoint,rag_qasper_lora_r16_seed42,sara_qasper_proj_lr5e4_seed42
2,Method Type,Standard RAG (Projector-free),SARA (LoRA + Projector)
3,Text Evidence (k),10,5
4,Compressed Evidence (n-k),0,5
5,LM LoRA LR,0.00002,0.00002
6,Projector LR,0.0,0.0005
7,Freeze Projector,True,False
8,Per Device Batch Size,1,1
9,Gradient Accumulation Steps,4,4


## 2. Dataset & Checkpoint Discovery Audit

In [3]:
dataset_split_path = REPO_ROOT / "data/reformatted/QASPER_train_split.jsonl"
dataset_manifest_path = REPO_ROOT / "data/manifests/qasper_split.json"

rag_ckpt_dir = REPO_ROOT / "checkpoints/finetune" / rag_cfg.get("output_model_name")
sara_ckpt_dir = REPO_ROOT / "checkpoints/finetune" / sara_cfg.get("output_model_name")

print("=" * 60)
print("DATASET & CHECKPOINT AUDIT STATUS")
print("=" * 60)
print(f"QASPER Train Split Exists : {dataset_split_path.exists()} ({dataset_split_path})")
print(f"QASPER Manifest Exists    : {dataset_manifest_path.exists()} ({dataset_manifest_path})")
print(f"Standard RAG Adapter Path : {rag_ckpt_dir.exists()} ({rag_ckpt_dir})")
print(f"SARA Adapter Path         : {sara_ckpt_dir.exists()} ({sara_ckpt_dir})")
print("=" * 60)

if dataset_manifest_path.exists():
    with open(dataset_manifest_path) as f:
        manifest = json.load(f)
    print(f"Manifest Papers Total: {manifest.get('n_papers_total')}, Train: {manifest.get('n_papers_train')}, Dev: {manifest.get('n_papers_dev')}")
    print(f"Train/Dev Paper Overlap: {manifest.get('overlap_train_dev')} (Zero Overlap Check: OK)")
print("=" * 60)

DATASET & CHECKPOINT AUDIT STATUS
QASPER Train Split Exists : True (/home/gunavenkat/Downloads/CS787-RAG-Project/Baselines/SARA-main/data/reformatted/QASPER_train_split.jsonl)
QASPER Manifest Exists    : True (/home/gunavenkat/Downloads/CS787-RAG-Project/Baselines/SARA-main/data/manifests/qasper_split.json)
Standard RAG Adapter Path : True (/home/gunavenkat/Downloads/CS787-RAG-Project/Baselines/SARA-main/checkpoints/finetune/rag_qasper_lora_r16_seed42)
SARA Adapter Path         : True (/home/gunavenkat/Downloads/CS787-RAG-Project/Baselines/SARA-main/checkpoints/finetune/sara_qasper_proj_lr5e4_seed42)
Manifest Papers Total: 858, Train: 772, Dev: 86
Train/Dev Paper Overlap: [] (Zero Overlap Check: OK)


## 3. Dataset Pipeline Construction (Run if Datasets are Missing)

In [4]:
if not dataset_split_path.exists():
    print("Building QASPER datasets and leakage-safe splits...")
    !CUDA_VISIBLE_DEVICES=0 {sys.executable} scripts/build_qasper_dataset.py
    !CUDA_VISIBLE_DEVICES=0 {sys.executable} -m src.data.make_qasper_splits --dev-frac 0.1 --seed 42
else:
    print("Dataset splits already exist. Ready for training or loading checkpoints.")

Dataset splits already exist. Ready for training or loading checkpoints.


## 4. Fine-Tuning Execution Commands

In [5]:
# --- Standard RAG Fine-Tuning ---
# Run this cell to train the Standard RAG adapter:
# !PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True CUDA_VISIBLE_DEVICES=0 {sys.executable} -m src.run_sara finetune --gpu 0 --config config/language_modeling/Finetune_RAG_qasper_pubbase.yaml

# --- SARA Fine-Tuning ---
# Run this cell to train the SARA adapter (LoRA + Projector):
# !PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True CUDA_VISIBLE_DEVICES=0 {sys.executable} -m src.run_sara finetune --gpu 0 --config config/language_modeling/Finetune_SARA_qasper_pubbase.yaml

print("Training commands prepared above. Uncomment and execute when ready to train.")

Training commands prepared above. Uncomment and execute when ready to train.


In [6]:
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True CUDA_VISIBLE_DEVICES=0 \
  /home/gunavenkat/Downloads/CS787-RAG-Project/sara_env/bin/python -m src.run_sara finetune \
  --gpu 0 --config config/language_modeling/Finetune_RAG_qasper_pubbase.yaml


INFO:datasets:PyTorch version 2.14.0 available.
[finetune] + CUDA_VISIBLE_DEVICES=0 /home/gunavenkat/Downloads/CS787-RAG-Project/sara_env/bin/python -m src.train.train_generator --config config/language_modeling/Finetune_RAG_qasper_pubbase.yaml
<frozen runpy>:128: RuntimeWarning: 'src.train.train_generator' found in sys.modules after import of package 'src.train', but prior to execution of 'src.train.train_generator'; this may result in unpredictable behaviour
Namespace(cache_dir='cache', chat_format='mistral', checkpointing_steps='epoch', chunk_size=256, compressor_name_or_path=None, config='config/language_modeling/Finetune_RAG_qasper_pubbase.yaml', data_dir='/home/gunavenkat/Downloads/CS787-RAG-Project/Baselines/SARA-main/data', dataset_name=None, debug=False, device=None, embedding_model=None, enable_fsdp=False, eval_steps=None, exclude_dataset_type=None, exp_name=None, exp_note=None, max_eval_samples=1000000000, max_seq_length=2048, max_train_samples=2000, max_train_steps=250, mix

In [7]:
!CUDA_VISIBLE_DEVICES=0 {sys.executable} -m src.run_sara finetune --gpu 0 --config config/language_modeling/Finetune_SARA_qasper_pubbase.yaml

INFO:datasets:PyTorch version 2.14.0 available.


[finetune] + CUDA_VISIBLE_DEVICES=0 /home/gunavenkat/Downloads/CS787-RAG-Project/sara_env/bin/python -m src.train.train_generator --config config/language_modeling/Finetune_SARA_qasper_pubbase.yaml


<frozen runpy>:128: RuntimeWarning: 'src.train.train_generator' found in sys.modules after import of package 'src.train', but prior to execution of 'src.train.train_generator'; this may result in unpredictable behaviour


Namespace(cache_dir='cache', chat_format='mistral', checkpointing_steps='epoch', chunk_size=256, compressor_name_or_path='Salesforce/SFR-Embedding-Mistral', config='config/language_modeling/Finetune_SARA_qasper_pubbase.yaml', data_dir='/home/gunavenkat/Downloads/CS787-RAG-Project/Baselines/SARA-main/data', dataset_name=None, debug=False, device=None, embedding_model=None, enable_fsdp=False, eval_steps=None, exclude_dataset_type=None, exp_name=None, exp_note=None, max_eval_samples=1000000000, max_seq_length=2048, max_train_samples=2000, max_train_steps=250, mixed_precision='fp16', num_workers=1, output_dir='/home/gunavenkat/Downloads/CS787-RAG-Project/Baselines/SARA-main/outputs', overwrite=False, qa_model=None, quantization=None, repetition_penalty=None, reranker_name_or_path='/workingdir/yjin328/SARA/checkpoints/reranker/mlp_epoch3.pth', retriever_name_or_path=None, seed=42, split_sentences=False, summarization_model=None, target_token=None, task_type='finetune', temperature=0.0, use_


Loading checkpoint shards:   0%|                         | 0/3 [00:00<?, ?it/s]

...


Tokenizing and reformatting data on rank: 0: 100%|█| 1996/1996 [00:28<00:00, 70

Filter:   0%|                                  | 0/1996 [00:00<?, ? examples/s]


Filter: 100%|████████████████████| 1996/1996 [00:00<00:00, 26361.83 examples/s]

Filter:   0%|                                  | 0/1996 [00:00<?, ? examples/s]


Filter: 100%|████████████████████| 1996/1996 [00:00<00:00, 22537.46 examples/s]


[W928 13:50:04.033544368 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 117440512 bytes (free: 62586880, total: 25182076928).
[W928 13:50:04.033636623 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 117440512 bytes (free: 62586880, total: 25182076928).


Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/home/gunavenkat/Downloads/CS787-RAG-Project/Baselines/SARA-main/src/train/train_generator.py", line 1330, in <module>
    main()
  File "/home/gunavenkat/Downloads/CS787-RAG-Project/Baselines/SARA-main/src/train/train_generator.py", line 1018, in main
    model, optimizer, train_dataloader, lr_scheduler = accelerator.prepare(
                                                       ^^^^^^^^^^^^^^^^^^^^
  File "/home/gunavenkat/Downloads/CS787-RAG-Project/sara_env/lib/python3.11/site-packages/accelerate/accelerator.py", line 1558, in prepare
    result = tuple(
             ^^^^^^
  File "/home/gunavenkat/Downloads/CS787-RAG-Project/sara_env/lib/python3.11/site-packages/accelerate/accelerator.py", line 1559, in <genexpr>
    self._prepare_one(obj, first_pass=True, device_placement=d) for obj, d in zip(args, device_placement)
    ^^^^^^^^^^^^^

## 5. Checkpoint Loader & Resource Profiler

In [7]:
from src.model.loader import load_sara_for_eval
import gc

def verify_and_profile_checkpoint(checkpoint_dir):
    ckpt_path = Path(checkpoint_dir)
    if not ckpt_path.exists():
        print(f"[WARNING] Checkpoint dir {checkpoint_dir} does not exist. Please train adapter first.")
        return None, None
    
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    
    t0 = time.time()
    ram_before = psutil.virtual_memory().used / (1024**3)
    
    print(f"Loading model checkpoint from: {checkpoint_dir}...")
    model, tokenizer = load_sara_for_eval(str(ckpt_path), device="cuda" if torch.cuda.is_available() else "cpu")
    
    load_time = time.time() - t0
    ram_after = psutil.virtual_memory().used / (1024**3)
    peak_vram = torch.cuda.max_memory_allocated() / (1024**3) if torch.cuda.is_available() else 0.0
    
    print("=" * 60)
    print(f"CHECKPOINT PROFILING SUMMARY")
    print("=" * 60)
    print(f"Checkpoint Path    : {ckpt_path.name}")
    print(f"Load Runtime       : {load_time:.2f} seconds")
    print(f"RAM Delta          : {ram_after - ram_before:.2f} GB")
    print(f"Peak GPU VRAM      : {peak_vram:.2f} GB")
    print(f"Compress Token ID  : {getattr(model, 'compress_token_id', None)}")
    print(f"Tokenizer Vocab    : {len(tokenizer)}")
    
    base_m = getattr(model, "get_base_model", lambda: model)()
    if hasattr(base_m, "projector") and base_m.projector is not None:
        proj_norm = sum(p.norm().item() for p in base_m.projector.parameters())
        print(f"Projector Param Norm: {proj_norm:.4f} (Non-zero check: OK)")
    else:
        print("Projector          : None (Projector-free Standard RAG)")
    print("=" * 60)
    return model, tokenizer

# Sequential profiling with explicit VRAM cleanup to prevent dual-7B CUDA OOM
print("--- 1/2 Profiling Standard RAG Checkpoint ---")
model_rag, tok_rag = verify_and_profile_checkpoint(str(rag_ckpt_dir))
del model_rag
torch.cuda.empty_cache()
gc.collect()

print("--- 2/2 Profiling SARA Checkpoint ---")
model_sara, tok_sara = verify_and_profile_checkpoint(str(sara_ckpt_dir))
del model_sara
torch.cuda.empty_cache()
gc.collect()


/home/gunavenkat/Downloads/CS787-RAG-Project/sara_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


--- 1/2 Profiling Standard RAG Checkpoint ---
Loading model checkpoint from: /home/gunavenkat/Downloads/CS787-RAG-Project/Baselines/SARA-main/checkpoints/finetune/rag_qasper_lora_r16_seed42...


`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 3/3 [00:00<00:00, 125.62it/s]
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
The new lm_head weights will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


CHECKPOINT PROFILING SUMMARY
Checkpoint Path    : rag_qasper_lora_r16_seed42
Load Runtime       : 10.23 seconds
RAM Delta          : 0.89 GB
Peak GPU VRAM      : 13.51 GB
Compress Token ID  : 32001
Tokenizer Vocab    : 32002
Projector          : None (Projector-free Standard RAG)
--- 2/2 Profiling SARA Checkpoint ---
Loading model checkpoint from: /home/gunavenkat/Downloads/CS787-RAG-Project/Baselines/SARA-main/checkpoints/finetune/sara_qasper_proj_lr5e4_seed42...


Loading checkpoint shards: 100%|██████████| 3/3 [00:00<00:00, 120.90it/s]
Some weights of XMistralForCausalLM were not initialized from the model checkpoint at mistralai/Mistral-7B-Instruct-v0.2 and are newly initialized: ['projector.projector.0.bias', 'projector.projector.0.weight', 'projector.projector.2.bias', 'projector.projector.2.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


CHECKPOINT PROFILING SUMMARY
Checkpoint Path    : sara_qasper_proj_lr5e4_seed42
Load Runtime       : 9.67 seconds
RAM Delta          : 0.06 GB
Peak GPU VRAM      : 13.58 GB
Compress Token ID  : 32001
Tokenizer Vocab    : 32002
Projector Param Norm: 164.0000 (Non-zero check: OK)


0

## 6. Single-Example Generation & Parity Test

In [ ]:
from src.model.xMistral import extract_generated_text

def run_sanity_generation_test(model, tokenizer, question="What are the key findings of this study?", evidence_texts=None):
    if model is None or tokenizer is None:
        print("Model or tokenizer not provided. Load checkpoint first.")
        return
    
    if evidence_texts is None:
        evidence_texts = [
            "The study demonstrates that selective context allocation reduces latency while maintaining QA accuracy.",
            "Context compression allows up to 10 passages to be represented efficiently in small token budgets."
        ]
    
    context_block = "".join(evidence_texts)
    user_prompt = f'''Context:
{context_block}

Question: {question}
Answer the question briefly based on the context.'''
    
    # Format with Mistral [INST] chat template
    if hasattr(tokenizer, "apply_chat_template") and tokenizer.chat_template:
        messages = [{"role": "user", "content": user_prompt}]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        prompt = f"<s>[INST] {user_prompt} [/INST]"
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
    
    t0 = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
            pad_token_id=pad_id,
            eos_token_id=tokenizer.eos_token_id
        )
    gen_time = time.time() - t0
    
    # XMistralForCausalLM returns ONLY newly generated tokens (no prompt prefix)
    gen_text_list = extract_generated_text(tokenizer, outputs)
    gen_text = gen_text_list[0] if gen_text_list else tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    print("=" * 60)
    print("GENERATION TEST OUTPUT")
    print("=" * 60)
    print(f"Generation Time : {gen_time:.2f} seconds")
    print(f"Generated Tokens: {outputs.shape[1]}")
    print(f"Generated Text  : {gen_text.strip()}")
    print("=" * 60)

# Example usage:
# model_rag, tok_rag = verify_and_profile_checkpoint(str(rag_ckpt_dir))
run_sanity_generation_test(model_rag, tok_rag)


GENERATION TEST OUTPUT
Generation Time : 1.82 seconds
Generated Tokens: 41
Generated Text  : The study shows that selective context allocation can reduce latency while maintaining question answering accuracy. Additionally, context compression allows up to 10 passages to be represented efficiently in small token budgets.


: 